Se creó una tabla adicional donde se guardaron únicamente las reservas canceladas de City Hotel, para poder trabajar con esos datos sin afectar la tabla original.


In [0]:
%sql
CREATE OR REPLACE TABLE cursodataacademy.hoteles.tb_reservas_prueba
USING DELTA
AS
SELECT *
FROM cursodataacademy.hoteles.hotel_bookings
WHERE 1 = 0;

INSERT INTO cursodataacademy.hoteles.tb_reservas_prueba
SELECT *
FROM cursodataacademy.hoteles.hotel_bookings
WHERE hotel = 'City Hotel'
  AND is_canceled = 1;

Se modificó la columna babies, colocando todos sus valores en cero, tal como se indicó en el problema de los registros incorrectos.

In [0]:
%sql

UPDATE cursodataacademy.hoteles.tb_reservas_prueba
SET babies = 0;

Se unieron las columnas del año, mes y día para obtener una fecha completa y más fácil de leer, sin tener que revisar cada dato por separado.

In [0]:
%sql

SELECT
    arrival_date_year,
    arrival_date_month,
    arrival_date_day_of_month,

    CONCAT(
        LPAD(CAST(arrival_date_day_of_month AS STRING), 2, '0'),
        '-',
        arrival_date_month,
        '-',
        CAST(arrival_date_year AS STRING)
    ) AS arrival_date_string

FROM cursodataacademy.hoteles.hotel_bookings;

Se filtraron los canales que comienzan con la letra T para identificar las reservas que ingresaron por este tipo de distribución

In [0]:
%sql

SELECT *
FROM cursodataacademy.hoteles.hotel_bookings
WHERE distribution_channel LIKE 'T%';

Se obtuvieron los datos generales de las reservas, las solicitudes especiales y el precio promedio por noche, teniendo así un resumen de la información del hotel.

In [0]:
%sql

SELECT
    COUNT(*) AS total_reservas,
    SUM(total_of_special_requests) AS total_solicitudes_especiales,
    AVG(adr) AS promedio_adr

FROM cursodataacademy.hoteles.hotel_bookings;

Se eliminaron de la tabla de prueba las reservas que tenían un ADR igual a cero, ya que estos registros fueron considerados pruebas del sistema.

In [0]:
%sql

DELETE FROM cursodataacademy.hoteles.tb_reservas_prueba
WHERE adr = 0.0;

Se agrupó la información por canal de distribución para conocer cuáles son los medios que generan una mayor cantidad de reservas.

In [0]:
%sql

SELECT
    distribution_channel,
    COUNT(*) AS total_reservas

FROM cursodataacademy.hoteles.hotel_bookings

GROUP BY distribution_channel

ORDER BY total_reservas DESC;

Se dividieron las reservas en tres categorías según su precio por noche, lo que permite diferenciar las habitaciones económicas, estándar y premium.

In [0]:
%sql

SELECT
    adr,

    CASE
        WHEN adr < 50 THEN 'Económico'
        WHEN adr BETWEEN 50 AND 150 THEN 'Estándar'
        WHEN adr > 150 THEN 'Premium'
        ELSE 'Sin clasificación'
    END AS categoria_precio

FROM cursodataacademy.hoteles.hotel_bookings;

Se calculó la fecha en la que se realizó cada reserva tomando como referencia la fecha de llegada y restando los días de anticipación.

In [0]:
%sql

SELECT
    arrival_date_year,
    arrival_date_month,
    arrival_date_day_of_month,
    lead_time,

    DATE_SUB(
        TO_DATE(
            CONCAT(
                arrival_date_day_of_month,
                '-',
                arrival_date_month,
                '-',
                arrival_date_year
            ),
            'd-MMMM-yyyy'
        ),
        CAST(lead_time AS INT)
    ) AS booking_date

FROM cursodataacademy.hoteles.hotel_bookings;

Se convirtió el lead_time de días a meses aproximados, utilizando CEIL para redondear los resultados hacia arriba.

In [0]:
%sql

SELECT
    lead_time,

    CEIL(lead_time / 30.0) AS meses_anticipacion_techo

FROM cursodataacademy.hoteles.hotel_bookings;